In [ ]:
import nomenclature
import pyam
import ixmp4

In [ ]:
from nomenclature.processor import DataValidator

In [ ]:
platform = ixmp4.Platform("scenariocompass-dev")

In [ ]:
capacity_validator = DataValidator.from_file(
    "validate_data/nearterm_capacity_expansion.yaml",
)

In [ ]:
variable_list = list()

for criterion in capacity_validator.criteria_items:
    variable_list.append(criterion.filter_args["variable"])

In [ ]:
criteria_names = list()

for criterion in capacity_validator.criteria_items:
    criteria_names.extend([criterion.name])

In [ ]:
filter_args = dict(
    variable=list(set(variable_list)),
    region="World",
)

In [ ]:
df = pyam.read_ixmp4(platform, **filter_args)

In [ ]:
for criterion in capacity_validator.criteria_items:
    
    #filter = criterion.filter_args
    #filter.pop("year")
    #df = pyam.read_ixmp4(platform, **filter)
    capacity_validator.apply(df)

In [ ]:
for criterion in capacity_validator.criteria_items:
    df.filter(year=range(2010, 2031), variable=criterion.filter_args["variable"]).plot(color=criterion.name, linewidth=0.3)

In [ ]:
for model, scenario in df.index:
    run = platform.runs.get(model=model, scenario=scenario)

    try:
        run.meta["Plausibility Vetting|Solar PV Capacity|World|2030"]
    except KeyError:
        for name in criteria_names:
            run.meta[name] = df.meta.loc[(model, scenario), name]
        print("Success " + model + " - " + scenario)